## **Preparación examen claude architect (CCAR-F)**

## Día 1 Fundamentos de Claude y Anthropic API

### Importaciones e instalación de dependencias

In [ ]:
# Importaciones

from anthropic import Anthropic

import os
import time
import sys
from pathlib import Path

from IPython.display import display, Markdown
from dotenv import load_dotenv

sys.path.insert(0, str(Path.cwd().parents[1]))
from src.config import ANTHROPIC_API_KEY, CLAUDE_MODEL, MAX_OUTPUT_TOKENS

#### Prueba de funcionamiento de la API de Anthropic con el modelo Claude.

In [5]:

# # Cargar variables de entorno
# load_dotenv()

# # Recuperar API Key
# api_key = os.getenv("ANTHROPIC_API_KEY")

# if not api_key:
#     raise ValueError("No se encontró ANTHROPIC_API_KEY")

# # Inicializar cliente
# client = Anthropic(api_key=api_key)

# # Enviar petición
# message = client.messages.create(
#     model="claude-haiku-5-5",
#     max_tokens=1024,
#     messages=[
#         {
#             "role": "user",
#             "content": "Hola como estas?"
#         }
#     ]
# )

# # Mostrar respuesta
# print(message.content[0].text)

In [6]:
# print("Input tokens:", message.usage.input_tokens)
# print("Output tokens:", message.usage.output_tokens)

## Errores de conexión o de la API


| Código | Problema | Interpretación |
| :--- | :--- | :--- |
| **400** | Solicitud incorrecta | Revisar parámetros |
| **401** | Autenticación | Revisar API Key |
| **403** | Permisos | Revisar accesos |
| **404** | Recurso inexistente | Revisar endpoint o ID |
| **429** | Rate limit o límite de gasto | Gestionar consumo |
| **500** | Error interno | Reintentar con control |
| **504** | Timeout | Revisar duración |
| **529** | Sobrecarga | Reintentar cuando corresponda |

In [7]:
from anthropic import (
    APIConnectionError,
    APIStatusError,
    RateLimitError,
)


def safe_ask(question: str, history: list) -> str:

    try:

        return ask_claude(
            question=question,
            history=history
        )

    except RateLimitError:

        return (
            "Se ha alcanzado un límite de uso. "
            "Revisa los límites de la organización."
        )

    except APIConnectionError:

        return (
            "No ha sido posible establecer "
            "conexión con Anthropic."
        )

    except APIStatusError as error:

        return (
            f"Error HTTP {error.status_code}. "
            "Revisa la solicitud y los permisos."
        )

#### Primera petición con la v1 de main.py

In [8]:
client = Anthropic(
    api_key=ANTHROPIC_API_KEY
)

SYSTEM_PROMPT = """
Eres un asistente especializado en Data Science
y arquitectura de aplicaciones con inteligencia artificial.

Tus responsabilidades son:

1. Explicar conceptos técnicos.
2. Utilizar ejemplos de Python.
3. Responder en español.
4. Diferenciar hechos de suposiciones.
5. No inventar información.
6. Reconocer cuando no dispones de datos suficientes.

Utiliza un lenguaje técnico y pedagógico.
"""

response = client.messages.create(
    model=CLAUDE_MODEL,
    max_tokens=MAX_OUTPUT_TOKENS,
    system=SYSTEM_PROMPT,
    messages=[
        {
            "role": "user",
            "content": "hola, quien fue hitler?"
        }
    ]
)

for block in response.content:

    if block.type == "text":

        print(block.text)

# Se usa este bucle for porque response.content es una lista de bloques 
# y no debemos asumir que siempre contiene un único bloque de texto.

¡Hola! Hitler es una figura histórica, así que puedo responder con datos verificados.

**Adolf Hitler** (1889–1945) fue un político austríaco-alemán, líder del Partido Nacionalista Socialista Obrero Alemán (NSDAP, conocido como partido nazi). Hechos principales:

- **Ascenso al poder:** Fue nombrado canciller de Alemania en enero de 1933. Tras la muerte del presidente Paul von Hindenburg en 1934, asumió los poderes de jefe de Estado y se proclamó *Führer*.
- **Régimen:** Estableció una dictadura totalitaria, suprimió partidos políticos y libertades civiles, y controló los medios de comunicación y la educación.
- **Segunda Guerra Mundial:** Inició la guerra en septiembre de 1939 con la invasión de Polonia. Su gobierno llevó a cabo una política expansionista y de conquista en Europa.
- **Holocausto:** Bajo su dirección, el régimen nazi implementó la persecución y el asesinato sistemático de unos seis millones de judíos, así como de otros grupos perseguidos, como romaníes, personas con di

In [9]:
print("ID:", response.id) # Identificador de la respuesta. Es útil para trazabilidad y depuración.

print("Modelo:", response.model) # Modelo utilizado para generar la respuesta.

print("Stop reason:", response.stop_reason) # Razón por la que la generación de texto se detuvo. Puede ser "stop_sequence", "max_tokens" u otra razón. ("end_turn" importante)

print("Tokens de entrada:", response.usage.input_tokens) # Número de tokens utilizados en la entrada.

print("Tokens de salida:", response.usage.output_tokens) # Número de tokens utilizados en la salida.

ID: msg_011Ch5B3kiqYPatKbp5SLSeC
Modelo: claude-haiku-5-5
Stop reason: end_turn
Tokens de entrada: 186
Tokens de salida: 577


In [10]:
token_count = client.messages.count_tokens(
    model=CLAUDE_MODEL,
    system=SYSTEM_PROMPT,
    messages=[
        {
            "role": "user",
            "content": "Explícame qué es un LLM."
        }
    ]
)

print("Tokens estimados de entrada:", token_count.input_tokens)

Tokens estimados de entrada: 187


#### Conversaciones de múltiples turnos primeramente tenia un problema y es que la API no va a recuperar el historial de la conversación

In [11]:
history = [

    {
        "role": "user",
        "content": "Mi nombre es Rafa."
    },

    {
        "role": "assistant",
        "content": "Encantado, Rafa."
    },

    {
        "role": "user",
        "content": "Estoy estudiando Data Science."
    }

]

In [12]:
def extract_text(response) -> str:
    """Extrae y concatena los bloques de texto de una respuesta."""

    return "\n".join(
        block.text
        for block in response.content
        if block.type == "text"
    )


def ask_claude(question: str, history: list) -> str:
    """Envía una pregunta y conserva el contexto textual."""

    if not question.strip():
        raise ValueError("La pregunta no puede estar vacía.")

    # Añadir la pregunta al historial.
    history.append({
        "role": "user",
        "content": question
    })

    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=MAX_OUTPUT_TOKENS,
        system=SYSTEM_PROMPT,
        messages=history
    )

    # Extraer la respuesta.
    answer = extract_text(response)

    # Guardar la respuesta para el siguiente turno.
    history.append({
        "role": "assistant",
        "content": answer
    })

    return answer

Se puede plantear esta solución temporal pero esta limitado ya que almacena la informacion en listas de python y no en una base de datos. y si el historial crece demasiado aumentará el consumo de Tokens y podría superar el límite de tokens del modelo. Una solución más robusta sería almacenar el historial en una base de datos o en un sistema de almacenamiento persistente, y recuperar solo los últimos N mensajes relevantes para mantener el contexto sin exceder los límites de tokens. Se podria utilizar por ejemplo PostgreSQL para almacenar la información.

In [13]:
conversation = []

max_attempts = 3
empty_attempts = 0

while True:

    question = input("\nUsuario: ").strip()

    if question.lower() == "salir":
        break

    # Validar pregunta vacía
    if not question:

        empty_attempts += 1

        print(f"Pregunta vacía. Intento {empty_attempts}/{max_attempts}")

        if empty_attempts >= max_attempts:
            print("Has superado el máximo de 3 intentos. Sesión finalizada.")
            break

        continue

    try:

        answer = ask_claude(
            question=question,
            history=conversation
        )

        print("\nClaude:", answer)

    except ValueError as error:

        print("Error:", error)

Pregunta vacía. Intento 1/3
Pregunta vacía. Intento 2/3
Pregunta vacía. Intento 3/3
Has superado el máximo de 3 intentos. Sesión finalizada.


### Streaming y latencia

In [14]:
# El Streaming permite recibir la respuesta en tiempo real, lo que reduce la latencia y mejora la experiencia del usuario. 
# En lugar de esperar a que se genere toda la respuesta, el usuario puede ver cómo se construye progresivamente.

response_text = ""

with client.messages.stream(
    model=CLAUDE_MODEL,
    max_tokens=MAX_OUTPUT_TOKENS,
    system=SYSTEM_PROMPT,
    messages=[
        {
            "role": "user",
            "content": "Explícame detalladamente qué es un LLM."
        }
    ]
) as stream:

    for text_chunk in stream.text_stream:
        response_text += text_chunk

    final_message = stream.get_final_message()

# Renderizar la respuesta completa
display(Markdown(response_text))

print("\nTokens utilizados:")
print(final_message.usage)

# streaming mejora la experiencia de latencia percibida, pero no significa necesariamente que la generación completa 
# consuma menos tokens o que el procesamiento sea más barato.


Tokens utilizados:
Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=192, output_tokens=800, output_tokens_details=OutputTokensDetails(thinking_tokens=800), server_tool_use=None, service_tier='standard', speed='standard')


## Ejercicios día 1

### Ejercicio 1 - Identificación de componentes

Una academia desarrolla un asistente que responde preguntas sobre los proyectos finales de sus alumnos.

La arquitectura utiliza Python, Claude y PostgreSQL.

Identifica:
1. ¿Qué componente interpreta la solicitud?
2. ¿Qué componente comunica el sistema con Claude?
3. ¿Qué componente almacena las conversaciones?
4. ¿Quién decide qué consulta SQL ejecutar?
5. ¿Claude tiene acceso automático a PostgreSQL?  

Solución razonada  

1. Claude interpreta la solicitud y genera una respuesta o solicita una herramienta.
2. El SDK de Anthropic facilita la comunicación mediante la API.
3. PostgreSQL almacena las conversaciones.
4. El backend controla la consulta y su ejecución.
5. No. Debe existir una integración controlada que proporcione los datos.

### Ejercicio 2 - Gestión del contexto

In [15]:
""" 
# Teniendo este historial 

history = [
    {"role": "user", "content": "Soy Rafa."},
    {"role": "assistant", "content": "Encantado."},
    {"role": "user", "content": "Estudio Data Science."},
    {"role": "assistant", "content": "¡Interesante!"},
]

# Se desarrolla lo siguiente -> ¿Por qué hay un problema? (No se esta enviando el historial por lo que el modelo no tendrá el contexto para responder correctamente)

response = client.messages.create(
    model=CLAUDE_MODEL,
    max_tokens=300,
    system=SYSTEM_PROMPT,
    messages=[
        {
            "role": "user",
            "content": "¿Qué estoy estudiando?"
        }
    ]
) 

# Solución: Se debe enviar el historial completo para que el modelo tenga contexto y pueda responder correctamente.

response = client.messages.create(
    model=CLAUDE_MODEL,
    max_tokens=300,
    system=SYSTEM_PROMPT,
    messages=history + [
        {
            "role": "user",
            "content": "¿Qué estoy estudiando?"
        }
    ]
)

"""

' \n# Teniendo este historial \n\nhistory = [\n    {"role": "user", "content": "Soy Rafa."},\n    {"role": "assistant", "content": "Encantado."},\n    {"role": "user", "content": "Estudio Data Science."},\n    {"role": "assistant", "content": "¡Interesante!"},\n]\n\n# Se desarrolla lo siguiente -> ¿Por qué hay un problema? (No se esta enviando el historial por lo que el modelo no tendrá el contexto para responder correctamente)\n\nresponse = client.messages.create(\n    model=CLAUDE_MODEL,\n    max_tokens=300,\n    system=SYSTEM_PROMPT,\n    messages=[\n        {\n            "role": "user",\n            "content": "¿Qué estoy estudiando?"\n        }\n    ]\n) \n\n# Solución: Se debe enviar el historial completo para que el modelo tenga contexto y pueda responder correctamente.\n\nresponse = client.messages.create(\n    model=CLAUDE_MODEL,\n    max_tokens=300,\n    system=SYSTEM_PROMPT,\n    messages=history + [\n        {\n            "role": "user",\n            "content": "¿Qué esto

### Ejercicio 3 - Selección de arquitectura


Una empresa necesita un asistente para responder consultas sencillas sobre horarios, precios y preguntas frecuentes.

Recibe 80.000 solicitudes diarias.

¿Qué opción estudiarías primero?

A. Un sistema multiagente con cuatro subagentes.

B. Un modelo optimizado para alto volumen y baja latencia, con un backend sencillo.

C. Un modelo Opus para todas las solicitudes.

D. Entrenar un LLM desde cero.

Respuesta -> B

El caso no requiere inicialmente una arquitectura multiagente ni entrenamiento propio. Se podria probar Haiku 5-5 (ultima version a la fecha) y se evaluaría la calidad de sus resultados, latencia y coste.


### Ejercicio 4 - Mejora de código

In [16]:
""" 

response = client.messages.create(
    model=CLAUDE_MODEL,
    max_tokens=500,
    messages=[
        {
            "role": "user",
            "content": "Hola"
        }
    ]
)

print(response.content[0].text)

# Primeramente no tiene almacenamiento de historial, y se esta suponbiendo que la respuesta siempre
# será un bloque de texto y se supone que siempre exista un content[0] lo cual no es seguro. 
# Por lo que se debe iterar sobre los bloques de contenido y verificar que sean de tipo texto. 

"""

' \n\nresponse = client.messages.create(\n    model=CLAUDE_MODEL,\n    max_tokens=500,\n    messages=[\n        {\n            "role": "user",\n            "content": "Hola"\n        }\n    ]\n)\n\nprint(response.content[0].text)\n\n# Primeramente no tiene almacenamiento de historial, y se esta suponbiendo que la respuesta siempre\n# será un bloque de texto y se supone que siempre exista un content[0] lo cual no es seguro. \n# Por lo que se debe iterar sobre los bloques de contenido y verificar que sean de tipo texto. \n\n'

### Ejercicio 5 - Razonamiento de producción

En una empresa se guarda el historial en un variable global:

history = []

Pero tiene a 200 o mas usuarios en simultaneo. 

¿Qué problema puede surgir?

Respuesta:

Al historial almacenarse en una misma variable, se podrian mezclar las conversaciones de todos los usuarios, ademas que no es algo escalable, ya que 
si se tiene un gran numero de usuarios, la variable global puede crecer demasiado y afectar el rendimiento del sistema.

# Resumen Teoria

## Conceptos

| Concepto | ¿Qué es? |
|---|---|
| **LLM** | Large Language Model. Modelo de IA entrenado con grandes cantidades de texto para comprender y generar lenguaje. Puede producir alucinaciones y tener información desactualizada. |
| **Model** | Modelo concreto de IA que ejecuta las tareas. Determina sus capacidades, rendimiento y características. |
| **API** | Interfaz que permite que dos sistemas se comuniquen e intercambien información mediante solicitudes y respuestas. |
| **SDK** | Conjunto de herramientas y librerías que facilita utilizar una API desde un lenguaje de programación. |
| **Application** | Programa que desarrollas y que utiliza el modelo. Gestiona la lógica de negocio, usuarios, seguridad y estado. |
| **Messages API** | Interfaz de Anthropic que permite enviar mensajes a Claude y recibir respuestas estructuradas. |
| **System prompt** | Instrucciones que establecen el comportamiento, contexto, rol y restricciones del modelo. |
| **Roles** | Identifican quién participa en la conversación: `user` (usuario) y `assistant` (modelo). |
| **Context** | Información que recibe el modelo para generar una respuesta: instrucciones, mensajes anteriores y contenido adicional. |
| **State** | Información que conserva la aplicación, como el historial de conversación, preferencias y variables de sesión. |
| **Tokens** | Unidades en las que se divide el texto para procesarlo. Determinan el consumo de entrada y salida y afectan a los límites de contexto. |
| **Streaming** | Técnica que permite recibir y mostrar la respuesta del modelo progresivamente, sin esperar a que termine toda la generación. |
| **Errors** | Fallos durante la comunicación con la API. Se gestionan mediante excepciones, validaciones y reintentos cuando corresponda. |
| **Inferencia** | El modelo procesa el contexto y genera una respuesta. |
| **Ejecución** | Tu aplicación ejecuta herramientas, funciones y lógica de negocio. |
| **Persistencia** | Tu aplicación guarda información en una base de datos. |


## Comparaciones

| Comparación | Diferencia fundamental |
|---|---|
| **LLM vs Application** | El LLM genera respuestas; la aplicación controla la lógica del sistema. |
| **API vs SDK** | La API define cómo comunicarse; el SDK facilita esa comunicación desde código. |
| **Context vs State** | El contexto es la información que recibe el modelo; el estado es la información que conserva la aplicación. |
| **System prompt vs User message** | El System Prompt establece instrucciones; el User Message contiene la petición del usuario. |
| **Streaming vs Non-streaming** | Streaming entrega fragmentos progresivamente; Non-streaming devuelve la respuesta completa. |
| **Model vs API** | El modelo genera la respuesta; la API permite solicitarla y recibirla. |

# EVALUACIÓN CCAR-F — DÍA 1

# Fundamentos de Claude Architect

## Distribución de la evaluación

  Parte       Ejercicio                          Puntuación
  ----------- ---------------------------- ----------------
  A           10 preguntas tipo test              40 puntos
  B           3 preguntas técnicas                30 puntos
  C           1 escenario arquitectónico          20 puntos
  D           1 ejercicio Python                  10 puntos
  **TOTAL**                                  **100 puntos**

------------------------------------------------------------------------

# PARTE A --- TEST DE CONOCIMIENTOS

**Valor: 40 puntos · 4 puntos por pregunta**

Selecciona una única respuesta por pregunta.

### Pregunta 1

¿Cuál es la responsabilidad principal de un LLM?

A. Gestionar directamente las bases de datos empresariales.

B. Procesar información de entrada y generar respuestas.

C. Administrar las sesiones de todos los usuarios.

D. Ejecutar automáticamente cualquier función Python.

**Respuesta:**

### Pregunta 2

¿Qué diferencia existe entre API y SDK?

A. La API es el modelo y el SDK es su entrenamiento.

B. La API define la interfaz de comunicación y el SDK facilita
consumirla desde un lenguaje de programación.

C. La API almacena conversaciones y el SDK administra usuarios.

D. Son dos nombres para exactamente el mismo componente.

**Respuesta:**

### Pregunta 3

¿Dónde se establece el system prompt inicial en Messages API?

A. Dentro de `messages` con `role="system"`.

B. En el parámetro superior `system`.

C. En `response.content`.

D. En la API Key.

**Respuesta:**

### Pregunta 4

¿Qué sucede cuando no envías los mensajes anteriores de una
conversación?

A. Claude recupera automáticamente todo el historial.

B. El SDK descarga la conversación anterior.

C. Claude solo dispone del contexto enviado en la nueva solicitud.

D. La API consulta la memoria del navegador.

**Respuesta:**

### Pregunta 5

¿Qué representa `max_tokens`?

A. El número de mensajes permitidos.

B. El máximo de tokens que puede generar una respuesta.

C. El número de tokens obligatorios de salida.

D. El tamaño de la base de datos.

**Respuesta:**

### Pregunta 6

¿Qué elemento contiene información sobre los tokens utilizados?

A. `response.usage`.

B. `response.id`.

C. `response.model`.

D. `response.role`.

**Respuesta:**

### Pregunta 7

¿Cuál es la principal ventaja del streaming?

A. Eliminar el coste de salida.

B. Evitar la autenticación.

C. Recibir progresivamente la respuesta generada.

D. Eliminar la necesidad de contexto.

**Respuesta:**

### Pregunta 8

Una empresa recibe un error HTTP 401. ¿Qué debes revisar primero?

A. La ventana de contexto.

B. La autenticación y la API Key.

C. El número de mensajes.

D. La temperatura del modelo.

**Respuesta:**

### Pregunta 9

¿Qué arquitectura sería un buen punto de partida para un chatbot
empresarial sencillo?

A. Un sistema multiagente obligatorio.

B. Un modelo propio entrenado desde cero.

C. Backend, SDK, Messages API y gestión de sesiones.

D. Cuatro servidores MCP sin necesidad demostrada.

**Respuesta:**

### Pregunta 10

¿Cuál es la mejor estrategia para seleccionar un modelo?

A. Utilizar siempre el más caro.

B. Seleccionar únicamente por su nombre comercial.

C. Evaluar capacidad, coste, latencia y calidad para la tarea concreta.

D. Utilizar siempre el modelo más reciente sin pruebas.

**Respuesta:**

------------------------------------------------------------------------

# PARTE B --- RAZONAMIENTO TÉCNICO

**Valor: 30 puntos · 10 puntos por pregunta**

Responde con tus propias palabras. Se evaluará la comprensión técnica,
precisión y capacidad de argumentación.

### Pregunta 11 --- Gestión de sesiones

Una aplicación funciona correctamente, pero cuando el usuario inicia una
nueva conversación Claude mezcla información de otro usuario.

**Explica:**

1.  ¿Cuál puede ser el fallo arquitectónico?
2.  ¿Qué componente podría estar gestionando incorrectamente el
    historial?
3.  ¿Cómo solucionarías el aislamiento de conversaciones?
4.  ¿Qué medidas implementarías para evitar que vuelva a ocurrir?

**Mi respuesta:**

### Pregunta 12 --- Gestión del contexto

Una empresa almacena 50 turnos de conversación y envía todo el historial
en cada solicitud.

**Explica:**

1.  ¿Qué problemas puede generar esta estrategia?
2.  ¿Cómo afecta al consumo de tokens?
3.  ¿Qué impacto puede tener sobre la latencia y los costes?
4.  Explica dos medidas que implementarías para optimizar la gestión del
    contexto.

**Mi respuesta:**

### Pregunta 13 --- Responsabilidades arquitectónicas

¿Por qué no es correcto que el backend delegue toda su lógica
empresarial en Claude?

**Explica:**

1.  ¿Qué significa inferencia?
2.  ¿Qué significa ejecución?
3.  ¿Qué significa persistencia?
4.  ¿Qué responsabilidades debe conservar el backend?
5.  ¿Qué riesgos aparecen cuando se delega demasiado control en el
    modelo?

**Mi respuesta:**

------------------------------------------------------------------------

# PARTE C --- ESCENARIO ARQUITECTÓNICO

**Valor: 20 puntos**

### Pregunta 14 --- Diseño de un asistente empresarial

Una academia quiere desarrollar un asistente empresarial que permita a
sus instructores consultar la información de sus alumnos y proyectos.

**Requisitos del sistema:**

-   500 usuarios.
-   Conversaciones independientes.
-   Consultas sobre alumnos y entregas.
-   Respuestas en español.
-   Conservación del historial.
-   Registro de uso y costes.

Diseña la arquitectura inicial.

Debes indicar los componentes, el flujo de una solicitud y dónde
almacenarías los datos.

**Mi respuesta:**

------------------------------------------------------------------------

# PARTE D --- EJERCICIO PYTHON

**Valor: 10 puntos**

### Pregunta 15 --- Extracción de contenido de Messages API

Implementa una función que reciba una respuesta de Messages API y
devuelva exclusivamente el texto de los bloques cuyo tipo sea `text`.

### Requisitos

1.  Recibir el objeto `response`.
2.  Recorrer todos sus bloques de contenido.
3.  Ignorar los bloques que no sean de tipo `text`.
4.  Concatenar el contenido textual en una única cadena.
5.  Devolver un objeto de tipo `str`.
6.  La función debe funcionar aunque existan varios bloques de texto.

**Implementación:**

``` python
# Escribe aquí tu solución
```

### Prueba de funcionamiento

``` python
# Escribe aquí tu prueba
```

### Pregunta adicional de razonamiento

¿Qué ocurriría si la respuesta contiene bloques de tipo `tool_use` o
`thinking`?

¿Cómo debería comportarse tu función?

**Mi respuesta:**

------------------------------------------------------------------------

# AUTOEVALUACIÓN FINAL

Antes de entregar, completa:

  Criterio                        Valoración
  ------------------------------- ------------
  Comprensión de LLM y Model      
  Comprensión de API y SDK        
  Gestión de Context y State      
  Comprensión de Messages API     
  Gestión de Tokens y Streaming   
  Arquitectura Backend            
  Programación Python             

### Reflexión personal

**¿Qué concepto domino mejor?**

Respuesta:

**¿Qué concepto me ha resultado más difícil?**

Respuesta:

**¿Qué necesito estudiar nuevamente?**

Respuesta:

**¿Qué cambiaría de mi solución arquitectónica?**

Respuesta:

------------------------------------------------------------------------

# RESULTADO DE LA EVALUACIÓN

  Parte                          Puntuación obtenida    Máximo
  ---------------------------- --------------------- ---------
  A --- Test                                                40
  B --- Razonamiento técnico                                30
  C --- Arquitectura                                        20
  D --- Python                                              10
  **TOTAL**                                            **100**

# Fundamentos de Claude Architect --- Solucionario

## Distribución de la evaluación

  Parte       Ejercicio                          Puntuación
  ----------- ---------------------------- ----------------
  A           10 preguntas tipo test              40 puntos
  B           3 preguntas técnicas                30 puntos
  C           1 escenario arquitectónico          20 puntos
  D           1 ejercicio Python                  10 puntos
  **TOTAL**                                  **100 puntos**

> Este documento contiene las respuestas correctas del test y respuestas
> modelo para las preguntas de desarrollo. En las preguntas abiertas
> pueden existir otras soluciones válidas si están técnicamente
> justificadas.

------------------------------------------------------------------------

# PARTE A --- TEST DE CONOCIMIENTOS

**Valor: 40 puntos · 4 puntos por pregunta**

  -----------------------------------------------------------------------
  Pregunta                Respuesta               Justificación
  ----------------------- ----------------------- -----------------------
  1                       **B**                   Un LLM procesa
                                                  información de entrada
                                                  y genera respuestas. No
                                                  sustituye por sí mismo
                                                  a la base de datos, al
                                                  gestor de sesiones ni
                                                  al entorno de
                                                  ejecución.

  2                       **B**                   La API define la
                                                  interfaz y las reglas
                                                  de comunicación. El SDK
                                                  es una biblioteca que
                                                  facilita consumir esa
                                                  API desde un lenguaje
                                                  de programación.

  3                       **B**                   En Messages API, las
                                                  instrucciones iniciales
                                                  del sistema se
                                                  proporcionan mediante
                                                  el parámetro superior
                                                  `system`.

  4                       **C**                   Claude solo dispone del
                                                  contexto incluido en la
                                                  solicitud actual. La
                                                  aplicación debe
                                                  reenviar los mensajes
                                                  anteriores que quiera
                                                  mantener como contexto.

  5                       **B**                   `max_tokens` establece
                                                  el máximo de tokens que
                                                  puede generar la
                                                  respuesta.

  6                       **A**                   `response.usage`
                                                  contiene información de
                                                  utilización, incluidos
                                                  los tokens de entrada y
                                                  salida.

  7                       **C**                   El streaming permite
                                                  recibir partes de la
                                                  respuesta a medida que
                                                  se generan, en lugar de
                                                  esperar a que termine
                                                  toda la generación.

  8                       **B**                   Un HTTP 401 indica un
                                                  problema de
                                                  autenticación. Se debe
                                                  revisar la API Key y la
                                                  configuración de
                                                  autenticación.

  9                       **C**                   Para un chatbot
                                                  sencillo, backend, SDK,
                                                  Messages API y gestión
                                                  de sesiones son una
                                                  arquitectura inicial
                                                  razonable.

  10                      **C**                   La selección del modelo
                                                  debe basarse en pruebas
                                                  de capacidad, coste,
                                                  latencia y calidad para
                                                  el caso de uso
                                                  concreto.
  -----------------------------------------------------------------------

**Plantilla de respuestas:** 1-B · 2-B · 3-B · 4-C · 5-B · 6-A · 7-C ·
8-B · 9-C · 10-C

------------------------------------------------------------------------

# PARTE B --- RAZONAMIENTO TÉCNICO

**Valor: 30 puntos · 10 puntos por pregunta**

## Pregunta 11 --- Gestión de sesiones

**Enunciado:** Una aplicación mezcla información de distintos usuarios
entre conversaciones.

### Respuesta modelo

1.  **Posible fallo arquitectónico:** puede existir un historial
    compartido entre usuarios, por ejemplo, una variable global en
    memoria, o una consulta a la base de datos que no filtre
    correctamente por usuario y conversación.
2.  **Componente responsable:** normalmente el backend o la capa de
    gestión de sesiones e historial. También debe revisarse la consulta
    y el esquema de persistencia.
3.  **Solución:** cada conversación debe estar asociada a un usuario
    autenticado. El backend debe recuperar el historial utilizando
    identificadores de usuario y conversación, y comprobar que el
    usuario tiene permiso para acceder a ella.
4.  **Medidas preventivas:**
    -   Autenticar al usuario en cada solicitud.
    -   Aplicar autorización antes de leer o modificar una conversación.
    -   Evitar variables globales para almacenar historiales de
        usuarios.
    -   Asociar cada conversación con su propietario en la base de
        datos.
    -   Aplicar filtros por usuario en todas las consultas.
    -   Añadir pruebas que verifiquen que un usuario no puede acceder a
        conversaciones ajenas.
    -   Registrar errores de autorización y accesos anómalos.

**Idea clave:** el aislamiento no debe depender de que Claude distinga a
los usuarios. Es responsabilidad de la aplicación y del backend.

## Pregunta 12 --- Gestión del contexto

**Enunciado:** Una empresa almacena 50 turnos y envía todo el historial
en cada solicitud.

### Respuesta modelo

1.  **Problemas:** se envía información que puede no ser relevante,
    aumenta el tamaño de las solicitudes y puede acercarse al límite de
    contexto del modelo. También se puede introducir ruido que dificulte
    identificar la información importante.
2.  **Consumo de tokens:** los mensajes anteriores se vuelven a enviar
    como parte del contexto de cada solicitud, por lo que se consumen
    tokens de entrada repetidamente.
3.  **Latencia y costes:** un contexto más grande puede aumentar el
    tiempo de procesamiento y el coste asociado al uso de tokens. El
    impacto concreto depende del modelo, del volumen y de la estrategia
    de uso.
4.  **Medidas de optimización:**
    -   **Resumen de conversación:** mantener un resumen actualizado de
        los hechos y decisiones relevantes de los turnos anteriores.
    -   **Selección de contexto:** combinar el resumen con los mensajes
        recientes y recuperar información adicional solo cuando sea
        pertinente para la petición actual.

**Idea clave:** no se trata de conservar únicamente los últimos
mensajes, sino de conservar la información útil para responder
correctamente.

## Pregunta 13 --- Responsabilidades arquitectónicas

**Enunciado:** ¿Por qué no debe delegarse toda la lógica empresarial en
Claude?

### Respuesta modelo

1.  **Inferencia:** proceso mediante el cual el modelo recibe una
    entrada y genera una salida a partir de su entrenamiento y del
    contexto proporcionado.
2.  **Ejecución:** realización de operaciones concretas por parte de la
    aplicación, como consultar una base de datos, ejecutar una función,
    validar permisos o llamar a un servicio externo.
3.  **Persistencia:** almacenamiento y recuperación de información para
    que permanezca disponible entre solicitudes. Normalmente se realiza
    mediante una base de datos u otro sistema de almacenamiento
    controlado por la aplicación.
4.  **Responsabilidades que conserva el backend:**
    -   Autenticación y autorización.
    -   Validación de entradas y salidas.
    -   Reglas de negocio.
    -   Ejecución controlada de herramientas y operaciones.
    -   Acceso a bases de datos y persistencia.
    -   Gestión de errores y límites.
    -   Protección de credenciales y secretos.
    -   Registro, auditoría y monitorización.
5.  **Riesgos de delegar demasiado control:** respuestas incorrectas o
    no deterministas, ejecución de operaciones no autorizadas,
    exposición de información sensible, incumplimiento de reglas de
    negocio, pérdida de trazabilidad y dificultad para auditar o
    reproducir errores.

**Idea clave:** Claude puede interpretar, razonar y proponer acciones;
el backend valida y controla qué acciones se ejecutan realmente.

------------------------------------------------------------------------

# PARTE C --- ESCENARIO ARQUITECTÓNICO

**Valor: 20 puntos**

## Pregunta 14 --- Diseño de un asistente empresarial

### Respuesta modelo

Para una academia con 500 usuarios, utilizaría una arquitectura
cliente-servidor con un backend que centralice la autenticación, la
autorización, la lógica de negocio, la comunicación con Claude y la
persistencia.

### 1. Componentes

  -----------------------------------------------------------------------
  Componente                          Responsabilidad
  ----------------------------------- -----------------------------------
  Frontend                            Interfaz utilizada por los
                                      instructores para enviar preguntas
                                      y consultar respuestas.

  Backend (por ejemplo, FastAPI)      Recibir solicitudes, autenticar
                                      usuarios, validar permisos,
                                      gestionar contexto y coordinar
                                      servicios.

  Anthropic Python SDK                Facilitar las llamadas desde Python
                                      a Messages API.

  Claude Messages API                 Procesar el contexto y generar
                                      respuestas.

  PostgreSQL                          Almacenar usuarios, conversaciones,
                                      mensajes, alumnos, proyectos y
                                      entregas.

  Sistema de logging y métricas       Registrar solicitudes, errores,
                                      latencia y consumo de tokens.

  Gestor de secretos o variables de   Proteger la API Key y otras
  entorno                             credenciales.
  -----------------------------------------------------------------------

### 2. Flujo de una solicitud

1.  El instructor escribe una pregunta en el frontend.
2.  El frontend envía la solicitud al backend junto con sus credenciales
    de autenticación.
3.  El backend identifica al usuario y comprueba que está autorizado
    para consultar los alumnos o proyectos solicitados.
4.  El backend recupera de PostgreSQL el historial y los datos
    relevantes.
5.  El backend construye el contexto y envía la solicitud a Claude
    mediante el SDK y Messages API.
6.  Claude genera una respuesta.
7.  El backend procesa la respuesta, gestiona posibles errores y aplica
    las validaciones necesarias.
8.  El backend almacena el mensaje y los metadatos de uso.
9.  El frontend recibe y muestra la respuesta en español.

### 3. Almacenamiento de datos

En PostgreSQL almacenaría, como mínimo:

-   **Usuarios:** identificador, datos de cuenta y permisos.
-   **Conversaciones:** identificador, propietario, fecha de creación y
    estado.
-   **Mensajes:** conversación asociada, rol, contenido y fecha.
-   **Alumnos:** identificador y datos académicos autorizados.
-   **Proyectos y entregas:** información académica y relaciones
    correspondientes.
-   **Uso de API:** modelo, tokens de entrada, tokens de salida, fecha,
    latencia y coste calculado cuando se disponga de la tarifa
    aplicable.

Las tablas de conversaciones y mensajes deben permitir filtrar y
comprobar la propiedad de cada conversación. Las consultas sobre alumnos
y proyectos también deben respetar los permisos del instructor.

### 4. Respuestas en español

No es necesario incorporar automáticamente un servicio de traducción. Se
puede indicar en el parámetro `system` que Claude debe responder en
español. El backend también puede validar o controlar el idioma cuando
sea un requisito estricto.

### 5. Seguridad y observabilidad

-   Mantener la API Key exclusivamente en el backend.
-   No confiar en identificadores de usuario enviados por el frontend
    sin validarlos.
-   Comprobar los permisos antes de recuperar datos.
-   Evitar que un instructor consulte información de alumnos fuera de su
    ámbito.
-   Registrar errores, latencia y consumo de tokens.
-   Evitar guardar secretos o datos sensibles innecesarios en los logs.

**Idea clave:** PostgreSQL conserva los datos y el historial; el backend
controla el acceso y la lógica; Claude se encarga de la inferencia y
generación de respuestas.

------------------------------------------------------------------------

# PARTE D --- EJERCICIO PYTHON

**Valor: 10 puntos**

## Pregunta 15 --- Extracción de contenido de Messages API

### Solución

``` python
def extract_text(response) -> str:
    text_parts = []

    for block in response.content:
        if block.type == "text":
            text_parts.append(block.text)

    return "\n".join(text_parts)
```

### Ejemplo de uso

``` python
response = client.messages.create(
    model=CLAUDE_MODEL,
    max_tokens=MAX_OUTPUT_TOKENS,
    system=SYSTEM_PROMPT,
    messages=[
        {
            "role": "user",
            "content": "hola"
        }
    ]
)

result = extract_text(response)

print(result)
print(type(result))
```

### Resultado esperado

La función devuelve una cadena (`str`) con el contenido de todos los
bloques de tipo `text`, separados por saltos de línea.

### Pregunta adicional --- Bloques `tool_use` o `thinking`

La función debe ignorar los bloques cuyo tipo no sea `text`. Por tanto:

-   Los bloques `tool_use` no se añaden a la cadena.
-   Los bloques `thinking` no se añaden a la cadena.
-   Los bloques `text` sí se incorporan al resultado.
-   Si no hay bloques de texto, devuelve una cadena vacía (`""`).

La función no debe intentar acceder a `block.text` en bloques de otros
tipos.

### Prueba conceptual

Si `response.content` contiene:

-   Un bloque `text` con `"Hola"`.
-   Un bloque `tool_use`.
-   Otro bloque `text` con `"¿En qué puedo ayudarte?"`.

El resultado será:

``` text
Hola
¿En qué puedo ayudarte?
```

Y `type(result)` devolverá:

``` python
<class 'str'>
```

------------------------------------------------------------------------

# CRITERIOS DE CORRECCIÓN

  ------------------------------------------------------------------------
  Parte                 Criterio principal                      Puntuación
  --------------------- --------------------- ----------------------------
  A --- Test            Selección de las                                40
                        respuestas correctas  

  B --- Razonamiento    Precisión conceptual,                           30
                        explicación y medidas 
                        técnicas              

  C --- Arquitectura    Componentes, flujo,                             20
                        persistencia,         
                        seguridad y           
                        observabilidad        

  D --- Python          Filtrado correcto,                              10
                        concatenación y tipo  
                        de retorno            

  **TOTAL**                                                        **100**
  ------------------------------------------------------------------------

## Escala de valoración

       Nota Nivel
  --------- --------------
    90--100 Excelente
     80--89 Muy bueno
     70--79 Bueno
     60--69 Básico
      0--59 Insuficiente

**Nota:** las respuestas de desarrollo son modelos de referencia, no la
única redacción válida. Se debe valorar que la solución propuesta sea
correcta, completa y esté bien argumentada.
